# 原始下一季营收指引：单信息增量

候选只增加一项原始管理层指导增长，控制组25价格特征、候选26特征使用相同日期/标签/来源可用mask。年度Ridge/prior选择、H5、费用10/25bp、风险约束均沿用固定方法；评估2016–2022开发及2023–2026复用诊断两窗。

计划为110次拟合、12个原生场景。MSE只以SOXL减SOXX的H5响应在两窗对两参照严格AND判定；IC只描述。原始指引不是latest-guide，不重构公司并购可比范围。源HTML字节与首次盘中秒级时点未认证，不签发生产。

In [1]:
from pathlib import Path
import os,sys,json,hashlib
import numpy as np,pandas as pd,yaml
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
ROOT=Path.cwd()
if not (ROOT/'src').is_dir():ROOT=ROOT.parents[1]
sys.path.insert(0,str(ROOT))
CARD=ROOT/'research/nvda_original_guidance_information_20261011'
plan=json.loads((CARD/'preregister_plan.json').read_text())
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
from src.core.recorder_run import start_evidence_run,publish_evidence_manifest
from src.data.provider_content_hash import provider_content_hash
from src.data.oos_horizon_inputs import prepare_horizon_inputs
from src.data.information_increment import load_information_state,build_matched_information_datasets
from src.model.generic_annual_ridge_selection import qualify_annual_horizon_inputs
from src.data.convex_allocation import LABEL_COLUMNS
qlib.init(provider_uri=str(ROOT/'data/qlib/published_futu_affine_20260925_v1'),region='us',kernels=1)
source_summary=json.loads((ROOT/plan['source_card']/'summary.json').read_text())
source_objects,source_audit=load_verified_recorder_objects(ROOT/plan['source_card'],expected_recorder_id=plan['source_recorder_id'],artifacts=['source/information_state.pkl','source/original_guidance_events.pkl'])
price=prepare_horizon_inputs(ROOT,json.loads((ROOT/plan['parent_plan']).read_text()))
information=load_information_state(provider_uri=ROOT/source_summary['source']['provider_uri'],content_sha256=source_summary['provider_content_sha256'],instrument=source_summary['source']['instrument'],field='$guidance_qoq',signal_dates=price['frame'].index)
bundle=build_matched_information_datasets(price,information,as_of=plan['as_of'])
assert (bundle['rows'],bundle['known_rows'],bundle['unknown_rows'])==(3956,3924,32)
qualification=[]
for year in plan['outer_years']:
    audits={}
    for arm in ['control','candidate']:
        audits[arm]=qualify_annual_horizon_inputs(bundle[arm],bundle['frame'],outer_year=year,
            feature_columns=bundle[arm+'_features'],label_columns=LABEL_COLUMNS,
            availability_column='h5_available_at',eligibility_column='source_known')['audit']
    for key in ['inner_identity','outer_identity','validation_identity']:
        assert audits['control'][key]==audits['candidate'][key]
    qualification.append(dict(audits['candidate'],both_arm_identities_equal=True))
assert qualification[0]['inner_training_rows']>=252
print('Native prefit3956/3924/32; all11annual sample/label identities equal.0fits.')


Native prefit3956/3924/32; all11annual sample/label identities equal.0fits.


In [2]:
if (CARD/'summary.json').exists():
    summary=json.loads((CARD/'summary.json').read_text())
    if summary['status']=='prefit_qualified_not_promoted':
        saved,audit=load_verified_recorder_objects(CARD,expected_recorder_id=summary['recorder_id'],artifacts=['prefit/qualification.pkl'])
        assert saved['prefit/qualification.pkl']==qualification
else:
    identity=start_evidence_run(experiment=plan['experiment'],uri=CARD/'qlib_recorder',name='native_matched_prefit',artifact_path='prefit',
        objects={'inputs.pkl':bundle,'qualification.pkl':qualification,'source_provenance.pkl':source_audit},
        params=dict(fit_calls=0,native_cases=0,fixed_future_fit_budget=110,preregister_sha256=hashlib.sha256((CARD/'preregister_plan.json').read_bytes()).hexdigest()))
    policy=dict(policy='slim_card_recorder_first',workflow_config='workflow.yaml',recorder_only_artifacts=['prefit/inputs.pkl','prefit/qualification.pkl','prefit/source_provenance.pkl'])
    summary=dict(identity,status='prefit_qualified_not_promoted',promotion_decision='not_promoted',promotion_eligible=False,fit_calls=0,native_cases=0,
        rows=3956,known_rows=3924,unknown_rows=32,recorder_status='FINISHED',source=source_summary['source'],artifact_policy=policy)
    publish_evidence_manifest(CARD,summary)
    (CARD/'workflow.yaml').write_text(yaml.safe_dump(dict(experiment=plan['experiment'],source=summary['source'],recorder_uri=summary['recorder_uri'],recorder_id=summary['recorder_id'],artifact_policy=policy),sort_keys=False))
    (CARD/'selected_model.json').write_text(json.dumps(dict(model=None,status='prefit_only',promotion_decision='not_promoted'),indent=2)+'\n')
    (CARD/'conclusion.md').write_text('已执行原生DatasetH和11个年度样本身份预检，3956日/3924已知/32未知；两组同日期同标签。0次拟合、0次回放，不晋升。随后仅允许冻结的110拟合/12原生场景，不做调参救结果。\n')
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "experiment": "nvda_original_guidance_information_20261011",
  "recorder_id": "6ec729a2e81348d697ee6478d7360f57",
  "recorder_uri": "/Users/rziiiii/dev/workspace/research/nvda_original_guidance_information_20261011/execution_recorder",
  "recorder_local_dir": "/Users/rziiiii/dev/workspace/research/nvda_original_guidance_information_20261011/execution_recorder/997153033184649364/6ec729a2e81348d697ee6478d7360f57",
  "status": "complete_not_promoted",
  "promotion_decision": "not_promoted",
  "promotion_eligible": false,
  "fit_calls": 110,
  "native_cases": 12,
  "historical_gate_pass": false,
  "diagnostic_gate_pass": false,
  "incremental_portfolio_pass": false,
  "incremental_information_pass": false,
  "route_decision": "end_original_guidance_route_no_further_search",
  "source": {
    "providers": {
      "price": {
        "provider_uri": "data/qlib/published_futu_affine_20260925_v1",
        "content_sha256": "b46e8edf02f46f15a75e07ded378e83bd13b6305fa98329754b23ea4cfec0efa"
 

In [3]:
from src.workflow.matched_information_experiment import run_matched_information_experiment
attempt=CARD/'execution_attempt.json'
if attempt.exists():
    state=json.loads(attempt.read_text())
    assert state['status']=='completed','blocked: preserve prior attempt; do not refit'
    summary=json.loads((CARD/'summary.json').read_text())
    saved,audit=load_verified_recorder_objects(CARD,expected_recorder_id=summary['recorder_id'],artifacts=['aggregate/final_summary.pkl'])
    assert saved['aggregate/final_summary.pkl']==summary
else:
    summary=run_matched_information_experiment(root=ROOT,card=CARD,execution_plan_path=CARD/'execution_plan.json')
print(json.dumps(summary,ensure_ascii=False,indent=2))
from IPython.display import Markdown,display
display(Markdown((CARD/'conclusion.md').read_text()))


{
  "experiment": "nvda_original_guidance_information_20261011",
  "recorder_id": "6ec729a2e81348d697ee6478d7360f57",
  "recorder_uri": "/Users/rziiiii/dev/workspace/research/nvda_original_guidance_information_20261011/execution_recorder",
  "recorder_local_dir": "/Users/rziiiii/dev/workspace/research/nvda_original_guidance_information_20261011/execution_recorder/997153033184649364/6ec729a2e81348d697ee6478d7360f57",
  "status": "complete_not_promoted",
  "promotion_decision": "not_promoted",
  "promotion_eligible": false,
  "fit_calls": 110,
  "native_cases": 12,
  "historical_gate_pass": false,
  "diagnostic_gate_pass": false,
  "incremental_portfolio_pass": false,
  "incremental_information_pass": false,
  "route_decision": "end_original_guidance_route_no_further_search",
  "source": {
    "providers": {
      "price": {
        "provider_uri": "data/qlib/published_futu_affine_20260925_v1",
        "content_sha256": "b46e8edf02f46f15a75e07ded378e83bd13b6305fa98329754b23ea4cfec0efa"
 

# 原始指引单信息增量：结果

固定 110 次拟合、12个原生现金/数量/费用场景完成。控制25价格特征与候选26特征的严格成熟日期、标签身份相同。

- development/10bp：净收益 351.88%，相对SOXX终值 1.061377，最大回撤 -41.21%。
- reused_diagnostic/10bp：净收益 426.61%，相对SOXX终值 1.057195，最大回撤 -42.79%。
- development/25bp：净收益 373.45%，相对SOXX终值 1.113700，最大回撤 -41.39%。
- reused_diagnostic/25bp：净收益 397.38%，相对SOXX终值 1.000000，最大回撤 -42.79%。

历史门 False；复用诊断门 False；对匹配控制组净组合增量 False；固定超额响应MSE增量 False。
不晋升；路线：end_original_guidance_route_no_further_search。
日期保守的原始季度指导量，不是最新公司预测。公司并购口径原样保留，不用未来公司实际或consensus。历史文本快照与秒级首次公开仍有限制；63个季度信息观测不等于3956个独立样本。
IC只是描述，不替代固定MSE及净超额门。费用改变优化动作时不同费用可能产生不同路径，不能解释为同路径加费改善。
失败即停止原始指引路线，不改符号/alpha/期限/阈值。即便历史门全过，仍需源身份与至少60日新独立前向证据。
Recorder 6ec729a2e81348d697ee6478d7360f57；FINISHED。


# 原始指引单信息增量：结果

固定 110 次拟合、12个原生现金/数量/费用场景完成。控制25价格特征与候选26特征的严格成熟日期、标签身份相同。

- development/10bp：净收益 351.88%，相对SOXX终值 1.061377，最大回撤 -41.21%。
- reused_diagnostic/10bp：净收益 426.61%，相对SOXX终值 1.057195，最大回撤 -42.79%。
- development/25bp：净收益 373.45%，相对SOXX终值 1.113700，最大回撤 -41.39%。
- reused_diagnostic/25bp：净收益 397.38%，相对SOXX终值 1.000000，最大回撤 -42.79%。

历史门 False；复用诊断门 False；对匹配控制组净组合增量 False；固定超额响应MSE增量 False。
不晋升；路线：end_original_guidance_route_no_further_search。
日期保守的原始季度指导量，不是最新公司预测。公司并购口径原样保留，不用未来公司实际或consensus。历史文本快照与秒级首次公开仍有限制；63个季度信息观测不等于3956个独立样本。
IC只是描述，不替代固定MSE及净超额门。费用改变优化动作时不同费用可能产生不同路径，不能解释为同路径加费改善。
失败即停止原始指引路线，不改符号/alpha/期限/阈值。即便历史门全过，仍需源身份与至少60日新独立前向证据。
Recorder 6ec729a2e81348d697ee6478d7360f57；FINISHED。

独立归因：全部12个原生场景SOXL数量为0，实际为SOXX/现金；开发25bp主要来自2022一次5日现金期，且与控制组相同。正超额年仅2/7、1/7，中位数/滚动/bootstrap均未通过，不能以累计收益或回撤改善宣称稳定超额。
